# ATAC cCRE PCA

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

# Resolve this analysis folder (paper/02_atac) regardless of the kernel's cwd.
def _here():
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / '01_ccre_insertion_matrix.py').exists():
            return c
    raise RuntimeError('run this notebook from inside paper/02_atac')

DIR = _here()
LCPM_PATH = DIR / 'results' / 'ccre_norm_lcpm.parquet'   # <- 02_normalize_loess.py
FIG_DIR = DIR / 'figs'
FIG_DIR.mkdir(parents=True, exist_ok=True)

if not LCPM_PATH.exists():
    raise SystemExit(f'missing {LCPM_PATH} — run 02_normalize_loess.py first')

SAMPLES = [
    '01_ESC.R1', '01_ESC.R2', '02_DE.R1', '02_DE.R2', '03_HB.R1', '03_HB.R2',
    '04_iHEP.R1', '04_iHEP.R2', '05_mHEP.R1', '05_mHEP.R2',
]
STAGES = ['ESC', 'DE', 'HB', 'iHEP', 'mHEP']
# Canonical project stage palette (matches 01_rna/data/stage_colors.json) so the
# ATAC and RNA PCAs colour-match.
STAGE_COLORS = {'ESC': '#2C6FB5', 'DE': '#00A878', 'HB': '#D4AC0D',
                'iHEP': '#EE5A24', 'mHEP': '#5E4327'}

## Load the normalized per-replicate matrix

In [ ]:
Y = pl.read_parquet(LCPM_PATH).select(SAMPLES).to_numpy()   # (n_ccre, 10)
print(f'{Y.shape[0]:,} expressed cCREs x {Y.shape[1]} replicates')

## PCA

Centered SVD on samples x features (matches sklearn `PCA`, no scaling).

In [ ]:
Xs = Y.T                                  # (10 samples, n_ccre)
Xc = Xs - Xs.mean(axis=0, keepdims=True)
U, Sv, _ = np.linalg.svd(Xc, full_matrices=False)
pcs = U * Sv                              # (10, 10)
ev = Sv ** 2 / (Sv ** 2).sum()

stage_of = [s.split('.')[0].split('_', 1)[1] for s in SAMPLES]   # ESC, DE, ...
rep_of = [s.split('.')[1] for s in SAMPLES]
print('PC var explained:', '  '.join(f'PC{i+1}={ev[i]:.1%}' for i in range(5)))

In [ ]:
FIG_NAME = '20_pca'

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
for ax, (a, b) in zip(axes, [(0, 1), (0, 2)]):
    for st in STAGES:
        idx = [i for i, s in enumerate(stage_of) if s == st]
        ax.scatter(pcs[idx, a], pcs[idx, b], c=STAGE_COLORS[st], label=st,
                   s=90, edgecolors='k', linewidths=0.5, zorder=3)
    for i in range(len(SAMPLES)):
        ax.annotate(rep_of[i], (pcs[i, a], pcs[i, b]), fontsize=6,
                    ha='center', va='center', zorder=4)
    ax.set_xlabel(f'PC{a+1} ({ev[a]:.1%})')
    ax.set_ylabel(f'PC{b+1} ({ev[b]:.1%})')
    ax.axhline(0, color='0.85', lw=0.5); ax.axvline(0, color='0.85', lw=0.5)
axes[0].set_title('PC1 vs PC2')
axes[1].set_title('PC1 vs PC3')
axes[0].legend(title='stage', fontsize=8)
fig.suptitle('ATAC cCRE PCA — loess-normalized log2-CPM (per replicate)',
             fontsize=12, y=1.0)
fig.tight_layout()

fig.savefig(FIG_DIR / f'{FIG_NAME}.png', dpi=150, bbox_inches='tight')
fig.savefig(FIG_DIR / f'{FIG_NAME}.pdf', bbox_inches='tight')
print('->', FIG_DIR / f'{FIG_NAME}.png')
fig